# DSA 495 — Module 08
## Convert PDFs to Text for RAG

Prepare the four supplied electricity-demand and electric-vehicle PDFs for a retrieval-augmented generation (RAG) knowledge base. Convert each PDF to a TXT file, inspect the extracted evidence, and identify information that text extraction may miss.

**Workflow:** PDFs → page text → TXT files → quality checks → RAG knowledge base.

Open this notebook in **Google Colab** and use a standard **CPU runtime**. Run cells from top to bottom. No model, GPU, or API key is needed.


## 1. Install and import the tools

PyMuPDF reads the existing text layer in each PDF. A scanned image needs optical character recognition (OCR); this notebook does not perform OCR. Charts and images are not converted into descriptions, and table structure may be lost.


In [ ]:
%pip install -q pymupdf==1.28.2


In [ ]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import pymupdf
from google.colab import drive, files
from IPython.display import Image, display

print("PyMuPDF version:", pymupdf.VersionBind)


## 2. Locate the course PDFs in Google Drive

Place the four PDFs from **M08_RAG02** in a folder in **My Drive**. Mount Drive and edit `pdf_folder` to match your folder's Colab path. The path below points to the course data folder; your own path may differ.

The supplied filenames begin with `D00000` through `D00003`. These identifiers remain in the TXT filenames so you can trace extracted evidence to the original PDF.


In [ ]:
drive.mount("/content/drive")


In [ ]:
pdf_folder = Path("/content/drive/MyDrive/a-ncsu-courses/DSA495-2026/Data/M08_RAG02")  # Edit to match your Drive folder.
pdf_paths = sorted(path for path in pdf_folder.iterdir()
                   if path.is_file() and path.suffix.lower() == ".pdf")

if not pdf_paths:
    raise ValueError("No PDFs found. Check pdf_folder and add the supplied PDFs.")

pd.DataFrame({"source_pdf": [path.name for path in pdf_paths]})


## 3. Extract and save the text

Read each PDF one page at a time. Within each text block, replace repeated whitespace and line breaks with spaces; separate blocks with blank lines. A block is a layout region and may not equal a paragraph. Sorting blocks helps with reading order but does not guarantee it.

The page-break character `\f` separates pages in each TXT file. This lets you recover a PDF page when checking the extraction. Headings, footers, and table text are retained for inspection.

Save the TXT files in a **text** subfolder beside the PDFs. Rerunning this section replaces the generated TXT files; the PDFs remain unchanged.

The summary flags pages with fewer than 80 extracted characters. A flagged page might be a cover, chart, divider, or scanned page. This is a cue for inspection, not an automatic diagnosis; unflagged pages can also contain errors.


In [ ]:
output_dir = pdf_folder / "text"
output_dir.mkdir(exist_ok=True)
text_paths = []
summary = []

for pdf_path in pdf_paths:
    page_texts = []
    low_text_pages = []
    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            blocks = page.get_text("blocks", sort=True)  # Extract text blocks in approximate reading order.
            paragraphs = [" ".join(block[4].split())
                          for block in blocks if block[6] == 0 and block[4].strip()]
            page_text = "\n\n".join(paragraphs)
            page_texts.append(page_text)
            if len(page_text.strip()) < 80:
                low_text_pages.append(page_number)

    if not any(text.strip() for text in page_texts):
        raise ValueError(f"{pdf_path.name}: no extractable text found. Inspect for blank or scanned pages; scanned pages need OCR.")

    text_path = output_dir / (pdf_path.stem + ".txt")
    text_path.write_text("\f".join(page_texts), encoding="utf-8")
    text_paths.append(text_path)
    summary.append({
        "source_pdf": pdf_path.name,
        "pages": len(page_texts),
        "words": sum(len(text.split()) for text in page_texts),
        "pages_to_check": ", ".join(map(str, low_text_pages)) or "None flagged",
    })

conversion_summary = pd.DataFrame(summary)
display(conversion_summary)
print("TXT folder:", output_dir)


## 4. Compare a PDF page with its extracted text

Start with page 3 of the first PDF. Change `pdf_to_check` to another file in the list and `page_to_check` to a page number between 1 and that PDF's page count. Inspect at least one flagged page and one page containing a table or chart.

PDF page numbers here count from the first page of the file; a page's printed number may differ. The text preview shows up to 4,000 characters.


In [ ]:
pdf_to_check = pdf_paths[0]
page_to_check = 3

with pymupdf.open(pdf_to_check) as pdf:
    page_image = pdf[page_to_check - 1].get_pixmap(dpi=100)
    display(Image(data=page_image.tobytes("png"), width=800))

text_to_check = output_dir / (pdf_to_check.stem + ".txt")
extracted_pages = text_to_check.read_text(encoding="utf-8").split("\f")
print(f"{pdf_to_check.name} — PDF page {page_to_check}")
print(extracted_pages[page_to_check - 1][:4000])


**Your turn:** Compare the page image with the extracted text.

- Are sentences and columns in a sensible reading order?
- Are headings, numbers, units, and qualifications preserved?
- Can you interpret a table from its extracted text? What does a chart show that the TXT file omits?

Record one example of readable evidence and one extraction limitation. Explain how that limitation could affect retrieval or a generated answer. A page with little text is not necessarily empty of useful information.


## 5. Use the TXT files as a RAG knowledge base

The TXT files are saved in `output_dir`. To use this collection with the Module 8 Gemma notebook, set that notebook's `KB_DIR` to the printed TXT folder path. Then rerun its document-loading, chunking, embedding, retrieval, and answer cells in order. That notebook requires a **T4 GPU** and Gemma access.

The RAG notebook will search these four documents instead of the Module 7 collection. Choose questions about their actual contents, such as electric-vehicle demand, large electricity loads, or planning uncertainty. For this collection, change the first line of `RAG_PROMPT` to “You are an evidence-based electricity-demand and energy-planning assistant.” Keep its evidence, citation, and missing-information instructions.

The RAG notebook cites filenames and chunk IDs; preserving page breaks here does not automatically add page-number citations there. Inspect the original PDF when a passage is unclear. This conversion step does not verify the sources' claims or produce reference answers.


### Optional: download the TXT files

Download a ZIP containing the TXT files created in this run. This also lets you move them to another knowledge-base folder.


In [ ]:
zip_path = Path("/content/DSA495-M08-PDF-Text.zip")
with ZipFile(zip_path, "w") as archive:
    for text_path in text_paths:
        archive.write(text_path, arcname=text_path.name)

files.download(str(zip_path))


### Reference

[PyMuPDF text-extraction documentation](https://pymupdf.readthedocs.io/en/latest/recipes-text.html) explains text blocks, reading order, and page separators.
